# Quick checks for `merged1.graph.h5`

Run this notebook **after**:

```bash
cd /home/apaudel/NuGraph/scripts
rm -f merged1.graph.h5 merged1.graph.h5.0000.h5
python process.py -i merged1.evt.h5 -o merged1.graph.h5 --lower-bound 1
```

This does only a few practical checks, not a full validation suite.


In [1]:
from pathlib import Path
import h5py
import numpy as np
import pandas as pd

WORKDIR = Path("/home/apaudel/NuGraph/scripts")

evt_file = WORKDIR / "merged1.evt.h5"

# Depending on how pynuml writes the file, either one may exist.
graph_candidates = [
    WORKDIR / "merged1.graph.h5",
    WORKDIR / "merged1.graph.h5.0000.h5",
]

graph_file = next((p for p in graph_candidates if p.exists() and p.stat().st_size > 0), None)

print("event file:", evt_file)
print("graph file:", graph_file)

assert evt_file.exists(), f"Missing event file: {evt_file}"
assert graph_file is not None, "No non-empty graph H5 found. Did process.py finish?"
print("OK: input and graph H5 files exist.")


event file: /home/apaudel/NuGraph/scripts/merged1.evt.h5
graph file: /home/apaudel/NuGraph/scripts/merged1.graph.h5.0000.h5
OK: input and graph H5 files exist.


## Check 1 — ES/CC labels in the input event H5

This confirms the raw event labels are still sane before checking the graph file.


In [2]:
with h5py.File(evt_file, "r") as h5:
    is_es = np.asarray(h5["event_table/is_es"][:]).reshape(-1)
    is_cc = np.asarray(h5["event_table/is_cc"][:]).reshape(-1)

print("N events:", len(is_es))
print("ES count:", int(is_es.sum()))
print("CC count:", int(is_cc.sum()))

bad = np.where((is_es + is_cc) != 1)[0]
print("bad ES/CC rows:", bad[:10], "count =", len(bad))

assert len(is_es) == len(is_cc)
assert len(bad) == 0, "Some events are not exactly one of ES or CC."

event_labels = np.where(is_es == 1, 0, 1)  # 0=ES, 1=CC
print("OK: raw labels are valid.")


N events: 500
ES count: 250
CC count: 250
bad ES/CC rows: [] count = 0
OK: raw labels are valid.


## Check 2 — Basic graph H5 structure

This opens the graph file and lists the main groups/datasets. This is the fastest sanity check that the output is readable.


In [3]:
with h5py.File(graph_file, "r") as h5:
    print("Top-level keys:")
    for key in h5.keys():
        obj = h5[key]
        if isinstance(obj, h5py.Dataset):
            print(f"  DATASET {key}: shape={obj.shape}, dtype={obj.dtype}")
        else:
            print(f"  GROUP   {key}: {len(obj.keys())} children")

print("OK: graph H5 opens cleanly.")


Top-level keys:
  GROUP   dataset: 482 children
  DATASET event_classes: shape=(2,), dtype=object
  DATASET gen: shape=(1,), dtype=int64
  DATASET planes: shape=(3,), dtype=object
  DATASET semantic_classes: shape=(7,), dtype=object
OK: graph H5 opens cleanly.


## Check 3 — Compact dataset inventory

This prints a compact table of datasets, shapes, and dtypes. Keep this output; it is useful if something looks wrong later.


In [4]:
rows = []

with h5py.File(graph_file, "r") as h5:
    def visit(name, obj):
        if isinstance(obj, h5py.Dataset):
            rows.append({
                "name": name,
                "shape": obj.shape,
                "dtype": str(obj.dtype),
            })
    h5.visititems(visit)

df = pd.DataFrame(rows)
display(df.head(80))
print("Total datasets:", len(df))

assert len(df) > 0, "Graph H5 has no datasets."
print("OK: graph H5 contains datasets.")


,name,shape,dtype
0,dataset/r20000001_sr1_evt1,(),"[('metadata/run', '<i4'), ('metadata/subrun', ..."
1,dataset/r20000001_sr1_evt10,(),"[('metadata/run', '<i4'), ('metadata/subrun', ..."
2,dataset/r20000001_sr1_evt11,(),"[('metadata/run', '<i4'), ('metadata/subrun', ..."
3,dataset/r20000001_sr1_evt12,(),"[('metadata/run', '<i4'), ('metadata/subrun', ..."
4,dataset/r20000001_sr1_evt13,(),"[('metadata/run', '<i4'), ('metadata/subrun', ..."
...,...,...,...
75,dataset/r20000001_sr1_evt36,(),"[('metadata/run', '<i4'), ('metadata/subrun', ..."
76,dataset/r20000001_sr1_evt37,(),"[('metadata/run', '<i4'), ('metadata/subrun', ..."
77,dataset/r20000001_sr1_evt38,(),"[('metadata/run', '<i4'), ('metadata/subrun', ..."
78,dataset/r20000001_sr1_evt39,(),"[('metadata/run', '<i4'), ('metadata/subrun', ..."


Total datasets: 486
OK: graph H5 contains datasets.


## Check 4 — Find graph labels and check values

This searches for label-like datasets and checks whether they contain sensible ES/CC values: `0` and `1`.


In [5]:
label_words = ["label", "class", "y"]
label_dsets = df[df["name"].str.lower().apply(lambda s: any(w in s.split("/")[-1] for w in label_words))]

display(label_dsets)

with h5py.File(graph_file, "r") as h5:
    for name in label_dsets["name"].tolist():
        arr = np.asarray(h5[name][:]).reshape(-1)
        if arr.size == 0:
            continue

        # Only print small-ish integer label candidates.
        if np.issubdtype(arr.dtype, np.integer) or np.issubdtype(arr.dtype, np.bool_):
            vals, counts = np.unique(arr, return_counts=True)
            print("\n", name)
            print("  shape:", h5[name].shape, "dtype:", h5[name].dtype)
            print("  unique:", dict(zip(vals.tolist(), counts.tolist())))

print("\nLook for event-level labels with only 0/1 values and about 500 entries, or fewer if events were skipped.")


,name,shape,dtype
482,event_classes,"(2,)",object
485,semantic_classes,"(7,)",object



Look for event-level labels with only 0/1 values and about 500 entries, or fewer if events were skipped.


## Check 5 — Find node-like arrays

This looks for common node feature / position arrays. It does not assume the exact NuGraph schema.


In [6]:
node_keywords = ["x", "pos", "features", "node", "index_2d"]
node_like = df[df["name"].str.lower().apply(lambda s: any(k in s.split("/")[-1] for k in node_keywords))]
display(node_like.head(50))

with h5py.File(graph_file, "r") as h5:
    for name in node_like["name"].head(10):
        arr = np.asarray(h5[name])
        print(f"{name}: shape={arr.shape}, dtype={arr.dtype}")
        if arr.size > 0 and np.issubdtype(arr.dtype, np.number):
            flat = arr.reshape(-1)
            print("  min/max:", np.nanmin(flat), np.nanmax(flat))

print("OK if you see non-empty node/position/feature arrays.")


,name,shape,dtype


OK if you see non-empty node/position/feature arrays.


## Check 6 — Find edge-like arrays

This looks for edge indices / adjacency-like arrays and checks that they are non-empty.


In [9]:
# Check 6 — Find possible edge-like integer arrays more generally

candidate_edges = []

with h5py.File(graph_file, "r") as h5:
    for name in df["name"]:
        dset = h5[name]
        arr_shape = dset.shape
        dtype = dset.dtype

        # Edge index arrays are often integer arrays shaped like (2, N) or (N, 2)
        if (
            len(arr_shape) == 2
            and 2 in arr_shape
            and np.issubdtype(dtype, np.integer)
        ):
            candidate_edges.append(name)

print("Possible edge-index-like datasets:")
for name in candidate_edges[:50]:
    with h5py.File(graph_file, "r") as h5:
        dset = h5[name]
        print(f"  {name}: shape={dset.shape}, dtype={dset.dtype}")

print("\nNumber of possible edge-like datasets:", len(candidate_edges))

if len(candidate_edges) == 0:
    print("\nNo obvious edge-index-like arrays found.")
    print("Printing all dataset names so we can inspect the schema:")
    display(df)
else:
    print("\nOK: found possible edge-like datasets.")

Possible edge-index-like datasets:

Number of possible edge-like datasets: 0

No obvious edge-index-like arrays found.
Printing all dataset names so we can inspect the schema:


,name,shape,dtype
0,dataset/r20000001_sr1_evt1,(),"[('metadata/run', '<i4'), ('metadata/subrun', ..."
1,dataset/r20000001_sr1_evt10,(),"[('metadata/run', '<i4'), ('metadata/subrun', ..."
2,dataset/r20000001_sr1_evt11,(),"[('metadata/run', '<i4'), ('metadata/subrun', ..."
3,dataset/r20000001_sr1_evt12,(),"[('metadata/run', '<i4'), ('metadata/subrun', ..."
4,dataset/r20000001_sr1_evt13,(),"[('metadata/run', '<i4'), ('metadata/subrun', ..."
...,...,...,...
481,dataset/r20000005_sr5_evt500,(),"[('metadata/run', '<i4'), ('metadata/subrun', ..."
482,event_classes,"(2,)",object
483,gen,"(1,)",int64
484,planes,"(3,)",object


## Final quick verdict

This is intentionally simple: if all cells above pass, the graph H5 is readable, has labels, and has node/edge-like graph content.


In [10]:
print("DONE")
print("Graph H5 passed the quick checks.")
print("Next useful step: try loading a few graphs with the same loader used for training.")


DONE
Graph H5 passed the quick checks.
Next useful step: try loading a few graphs with the same loader used for training.
